# Teoria da Decisão - ENTREGA #1: Otimização Mono-objetivo

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. Carregamento de Dados

Carregamos os arquivos `distancia.csv` e `tempo.csv` para matrizes NumPy. Estas matrizes serão usadas como as funções de custo para o nosso problema de otimização.

In [ ]:
try:
    distancia_df = pd.read_csv('distancia.csv', header=None)
    distancia_matrix = distancia_df.values

    tempo_df = pd.read_csv('tempo.csv', header=None)
    tempo_matrix = tempo_df.values

    print(f"Matriz de distâncias carregada com sucesso. Shape: {distancia_matrix.shape}")
    print(f"Matriz de tempos carregada com sucesso. Shape: {tempo_matrix.shape}")
except FileNotFoundError as e:
    print(f"Erro: Arquivo não encontrado. Verifique se os arquivos .csv estão no diretório correto. {e}")

## 2. Função de Custo e Representação da Solução

Uma solução (ou rota) é representada por um array de inteiros que define a ordem em que as cidades são visitadas. A função de custo calcula o comprimento total de uma rota, somando os custos (distância ou tempo) de cada segmento da viagem, incluindo o retorno à cidade inicial.

In [ ]:
def calcular_custo(rota, matriz_custo):
    """Calcula o custo total de uma rota."""
    custo_total = 0
    for i in range(len(rota) - 1):
        custo_total += matriz_custo[rota[i], rota[i+1]]
    custo_total += matriz_custo[rota[-1], rota[0]]  # Retorno à cidade inicial
    return custo_total

## 3. Heurística Construtiva para a Solução Inicial

Para gerar uma solução inicial de boa qualidade, usamos uma heurística construtiva gulosa e aleatorizada. O algoritmo começa em uma cidade aleatória e, a cada passo, adiciona a cidade mais próxima que ainda não foi visitada. Para introduzir variabilidade, a escolha da "cidade mais próxima" é feita a partir de uma lista restrita de candidatos (RCL - Restricted Candidate List), e não apenas da melhor opção.

In [ ]:
def heuristica_construtiva_gulosa_aleatorizada(matriz_custo, rcl_size=3):
    """Gera uma solução inicial usando uma heurística gulosa aleatorizada."""
    num_cidades = matriz_custo.shape[0]
    cidade_inicial = np.random.randint(num_cidades)
    rota = [cidade_inicial]
    cidades_nao_visitadas = list(range(num_cidades))
    cidades_nao_visitadas.remove(cidade_inicial)

    cidade_atual = cidade_inicial
    while cidades_nao_visitadas:
        custos = [(matriz_custo[cidade_atual, proxima_cidade], proxima_cidade) for proxima_cidade in cidades_nao_visitadas]
        custos.sort()
        
        # Seleciona um candidato da RCL
        rcl = custos[:rcl_size]
        _, proxima_cidade = rcl[np.random.randint(len(rcl))]
        
        rota.append(proxima_cidade)
        cidades_nao_visitadas.remove(proxima_cidade)
        cidade_atual = proxima_cidade
        
    return np.array(rota)

## 4. Estruturas de Vizinhança

Implementamos três estruturas de vizinhança para explorar o espaço de soluções:
*   **2-Opt:** Uma troca simples de duas arestas para remover cruzamentos na rota.
*   **Or-Opt:** Move um pequeno bloco de cidades para uma posição diferente na rota.
*   **Double-Bridge:** Uma perturbação maior que corta a rota em quatro partes e as reconecta de uma maneira diferente.

In [ ]:
def vizinhanca_2_opt(rota):
    """Gera uma nova rota trocando duas arestas."""
    num_cidades = len(rota)
    i, j = np.random.choice(num_cidades, 2, replace=False)
    if i > j: i, j = j, i
    nova_rota = np.copy(rota)
    nova_rota[i:j+1] = np.flip(nova_rota[i:j+1])
    return nova_rota

def vizinhanca_or_opt(rota, tamanho_bloco=3):
    """Move um bloco de cidades para outra posição."""
    num_cidades = len(rota)
    i = np.random.randint(0, num_cidades - tamanho_bloco)
    j = np.random.randint(0, num_cidades - tamanho_bloco)
    
    bloco = rota[i:i+tamanho_bloco]
    rota_sem_bloco = np.delete(rota, np.arange(i, i + tamanho_bloco))
    
    return np.insert(rota_sem_bloco, j, bloco)

def vizinhanca_double_bridge(rota):
    """Aplica o movimento double-bridge para perturbar a rota."""
    num_cidades = len(rota)
    indices = sorted(np.random.choice(num_cidades, 4, replace=False))
    i, j, k, l = indices
    
    parte1 = rota[:i]
    parte2 = rota[i:j]
    parte3 = rota[j:k]
    parte4 = rota[k:l]
    parte5 = rota[l:]
    
    return np.concatenate([parte1, parte4, parte3, parte2, parte5])

## 5. Busca Local (Descida em Vizinhança Variável - VND)

O VND é um método de busca local que explora sistematicamente várias estruturas de vizinhança. Se uma melhora é encontrada em uma vizinhança, a busca é reiniciada a partir da primeira vizinhança. Se não, a busca continua com a próxima vizinhança da lista.

In [ ]:
def vnd(rota_inicial, matriz_custo, vizinhancas):
    """Aplica o Variable Neighborhood Descent para refinar uma solução."""
    melhor_rota = np.copy(rota_inicial)
    melhor_custo = calcular_custo(melhor_rota, matriz_custo)
    
    l = 0
    while l < len(vizinhancas):
        vizinhanca = vizinhancas[l]
        nova_rota = vizinhanca(melhor_rota)
        novo_custo = calcular_custo(nova_rota, matriz_custo)
        
        if novo_custo < melhor_custo:
            melhor_rota = nova_rota
            melhor_custo = novo_custo
            l = 0
        else:
            l += 1
            
    return melhor_rota, melhor_custo

## 6. Metaheurística (Busca em Vizinhança Variável Geral - GVNS)

O GVNS é a metaheurística principal que orquestra a busca. Ele utiliza o VND para refinar soluções e as estruturas de vizinhança para perturbar a solução (shaking) e escapar de ótimos locais.

In [ ]:
def gvns(matriz_custo, vizinhancas, kmax, max_iter, return_curve=False):
    """Executa o General Variable Neighborhood Search."""
    # Solução inicial
    rota_inicial = heuristica_construtiva_gulosa_aleatorizada(matriz_custo)
    melhor_rota, melhor_custo = vnd(rota_inicial, matriz_custo, vizinhancas)
    
    historico_custos = [melhor_custo]
    iter = 0
    
    while iter < max_iter:
        k = 0
        while k < kmax:
            # Shaking
            vizinhanca_shake = vizinhancas[k]
            rota_perturbada = vizinhanca_shake(melhor_rota)
            
            # Busca local
            nova_rota, novo_custo = vnd(rota_perturbada, matriz_custo, vizinhancas)
            
            if novo_custo < melhor_custo:
                melhor_rota = nova_rota
                melhor_custo = novo_custo
                k = 0
                iter = 0
            else:
                k += 1
                iter += 1
            
            historico_custos.append(melhor_custo)
            
    if return_curve:
        return melhor_rota, melhor_custo, historico_custos
    else:
        return melhor_rota, melhor_custo

## 7. Execução e Resultados da Otimização Mono-objetivo

Executamos o GVNS 5 vezes para cada função objetivo (tempo e distância) e apresentamos os resultados.

In [ ]:
# Parâmetros
KMAX = 3
TMAX = 1000
NUM_EXECUCOES = 5
vizinhancas = [vizinhanca_2_opt, vizinhanca_or_opt, vizinhanca_double_bridge]

resultados = {
    'Tempo': [],
    'Distancia': [],
    'Historico_Tempo': [],
    'Historico_Distancia': []
}

# Otimização para Tempo
print("Otimizando para Tempo...")
for i in range(NUM_EXECUCOES):
    _, custo, historico = gvns(tempo_matrix, vizinhancas, KMAX, TMAX, return_curve=True)
    resultados['Tempo'].append(custo)
    resultados['Historico_Tempo'].append(historico)
    print(f"Execução {i+1}/5 (Tempo): Custo = {custo}")

# Otimização para Distância
print("\nOtimizando para Distância...")
for i in range(NUM_EXECUCOES):
    _, custo, historico = gvns(distancia_matrix, vizinhancas, KMAX, TMAX, return_curve=True)
    resultados['Distancia'].append(custo)
    resultados['Historico_Distancia'].append(historico)
    print(f"Execução {i+1}/5 (Distância): Custo = {custo}")

# Apresentação dos Resultados
df_resultados = pd.DataFrame(
    {
        'Função': ['Tempo', 'Distancia'],
        'Mínimo': [np.min(resultados['Tempo']), np.min(resultados['Distancia'])],
        'Máximo': [np.max(resultados['Tempo']), np.max(resultados['Distancia'])],
        'Desvio Padrão': [np.std(resultados['Tempo']), np.std(resultados['Distancia'])]
    }
    )

print("\nResultados da Otimização Mono-objetivo:")
print(df_resultados.to_string(index=False))

## 8. Gráficos de Convergência

Os gráficos abaixo mostram a convergência do algoritmo GVNS para cada uma das 5 execuções, para as funções objetivo de tempo e distância.

In [ ]:
# Gráfico de Convergência para Tempo
plt.figure(figsize=(12, 6))
for i, historico in enumerate(resultados['Historico_Tempo']):
    plt.plot(historico, label=f'Execução {i+1}')
plt.title('Convergência do GVNS para o Tempo')
plt.xlabel('Iterações')
plt.ylabel('Custo (Tempo)')
plt.legend()
plt.grid(True)
plt.show()

# Gráfico de Convergência para Distância
plt.figure(figsize=(12, 6))
for i, historico in enumerate(resultados['Historico_Distancia']):
    plt.plot(historico, label=f'Execução {i+1}')
plt.title('Convergência do GVNS para a Distância')
plt.xlabel('Iterações')
plt.ylabel('Custo (Distância)')
plt.legend()
plt.grid(True)
plt.show()